In [26]:
!pip install -q groq

In [27]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = None

if not GROQ_API_KEY:
    GROQ_API_KEY = getpass("Enter Groq API key: ")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("Groq API key configured!")

Enter Groq API key: ··········
Groq API key configured!


In [29]:
from groq import Groq
import os
import time

client = Groq(
    api_key=os.environ["GROQ_API_KEY"],
    timeout=30.0,
    max_retries=1
)

MODEL = "openai/gpt-oss-20b"

start = time.time()

try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": "Explain Python variables in simple English."
            }
        ],
        max_completion_tokens=300,
        temperature=0.5
    )

    print("SUCCESS!")
    print("\nResponse:")
    print(response.choices[0].message.content)

    print("\nTime taken:", round(time.time() - start, 2), "seconds")

except Exception as e:
    print("ERROR:", repr(e))

SUCCESS!

Response:
### What is a Python variable?

Think of a **variable** as a *label* or *name* that you

Time taken: 0.52 seconds


In [30]:
from groq import Groq
import os
import time

client = Groq(
    api_key=os.environ["GROQ_API_KEY"],
    timeout=30.0,
    max_retries=1
)

MODEL = "openai/gpt-oss-20b"

try:
    start = time.time()

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are Byte, a friendly programming mentor."
            },
            {
                "role": "user",
                "content": "Explain Python variables in simple English."
            }
        ],
        max_completion_tokens=500
    )

    print("SUCCESS!")
    print("\nByte:")
    print(response.choices[0].message.content)

    print("\nTime:", round(time.time() - start, 2), "seconds")

except Exception as e:
    print("ERROR:", repr(e))

SUCCESS!

Byte:
## Python Variables – The “Name Box” of Your Code

Imagine you’re working with a bunch of labeled boxes.  
Each box can hold a value (a number, a word, a picture, etc.).  
In Python, **variables** are the labels on those boxes.

---

### 1. What is a variable?

- **A name that points to a value.**  
  When you write `age = 25`, you’re telling Python:  
  *“Create a box called `age` and put the number 25 inside it.”*

- **No fixed type.**  
  Python doesn’t need to know *what* kind of value the box will hold until you put something in it.  
  ```python
  age = 25          # integer
  name = "Alice"    # string
  age = 25.4        # now it's a float, same box, different content
  ```

---

### 2. Basic Rules for Naming

| Rule | Example | Explanation |
|------|---------|-------------|
| **Start with a letter or underscore** | `first_name` | Cannot start with a number (`3dogs` is invalid). |
| **Only use letters, numbers, underscore** | `my_var2` | No spaces, hyphens, or p

In [31]:
import os
from getpass import getpass
from groq import Groq
import groq
import time

# Retrieve API key from Colab Secrets
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = None

# Fallback
if not GROQ_API_KEY:
    GROQ_API_KEY = getpass("Enter your Groq API key: ")

if not GROQ_API_KEY:
    raise ValueError("Groq API key is required!")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Initialize Groq client
client = Groq(
    api_key=GROQ_API_KEY,
    timeout=30.0,
    max_retries=2
)

# Configuration
MODEL = "openai/gpt-oss-20b"

MAX_RECENT = 10
SUMMARIZE_AFTER = 16

MAX_USER_CHARS = 4000
MAX_OUTPUT_TOKENS = 2048

print("Byte chatbot configuration completed!")
print("Model:", MODEL)

Enter your Groq API key: ··········
Byte chatbot configuration completed!
Model: openai/gpt-oss-20b


In [32]:
SYSTEM_PROMPT = """
You are Byte, a friendly and patient programming mentor
for beginners.

ROLE AND PURPOSE:
- Help people learn Python, JavaScript and computer science.
- Explain programming concepts step by step.
- Help beginners understand difficult topics.
- Provide practical coding examples.
- Help users debug programming errors.

PERSONALITY:
- Friendly, encouraging and patient.
- Never make learners feel uncomfortable.
- Use simple and clear English.
- Explain complicated concepts in easy language.

TEACHING STYLE:
- Break complicated concepts into smaller steps.
- Use real-world examples.
- Explain the logic behind code.
- Use Markdown formatting.
- Use code blocks for programming examples.
- Be concise unless the user asks for more detail.

CONVERSATION RULES:
- Remember information shared in earlier messages.
- Maintain continuity across multiple turns.
- Refer to previous discussions when relevant.
- Never ask users to repeat information already provided.

CONSTRAINTS:
- Focus primarily on programming and technology.
- Never invent programming functions or libraries.
- Admit uncertainty when necessary.
"""

SUMMARY_PROMPT = """
You are a conversation summarization assistant.

Your task is to maintain a running summary of a conversation
between a user and a programming mentor chatbot.

Combine the existing summary with the new messages.

PRESERVE:
1. User's name and learning goals.
2. Programming languages discussed.
3. Important facts shared by the user.
4. Relevant previous questions and answers.
5. Decisions and unresolved questions.

REMOVE:
1. Greetings.
2. Repeated information.
3. Unnecessary details.

Keep the summary within 150 words.

Return ONLY the updated summary.
"""

print("System prompt and summary prompt configured!")

System prompt and summary prompt configured!


In [33]:
# ============================================================
# BYTE: MULTI-TURN LLM CHATBOT
# ============================================================

class ChatError(Exception):
    """Custom exception for chatbot errors."""
    pass


class Chatbot:

    def __init__(self, conversation_id="default"):

        self.conversation_id = conversation_id

        # Stores recent conversation messages
        self.messages = []

        # Stores summary of older conversation messages
        self.summary = ""

        print(f"Conversation '{conversation_id}' initialized!")


    # ========================================================
    # 1. LLM API CALL WITH ERROR HANDLING
    # ========================================================

    def _call_llm(self, messages, max_tokens=MAX_OUTPUT_TOKENS):

        try:

            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=0.7,
                max_completion_tokens=max_tokens
            )

            if not response.choices:
                raise ChatError("The AI returned no response.")

            reply = response.choices[0].message.content

            if not reply or not reply.strip():
                raise ChatError("The AI returned an empty response.")

            return reply.strip()


        except groq.AuthenticationError:

            raise ChatError(
                "Authentication failed. Check your Groq API key."
            )


        except groq.RateLimitError:

            raise ChatError(
                "Groq API rate limit reached. Please wait and try again."
            )


        except groq.PermissionDeniedError:

            raise ChatError(
                "Your API key does not have permission to use this model."
            )


        except groq.NotFoundError:

            raise ChatError(
                f"Model '{MODEL}' is unavailable. Check model access."
            )


        except groq.APITimeoutError:

            raise ChatError(
                "The AI request timed out. Please try again."
            )


        except groq.APIConnectionError:

            raise ChatError(
                "Unable to connect to Groq. Check your internet connection."
            )


        except groq.InternalServerError:

            raise ChatError(
                "Groq service is temporarily unavailable. Try again later."
            )


        except groq.APIStatusError as e:

            raise ChatError(
                f"API request failed. Status code: {e.status_code}"
            )


        except ChatError:
            raise


        except Exception as e:

            raise ChatError(
                f"Unexpected error: {type(e).__name__}"
            )


    # ========================================================
    # 2. CONTEXT WINDOW MANAGEMENT
    # ========================================================

    def _compact(self):

        # Summarize only when the threshold is exceeded
        if len(self.messages) <= SUMMARIZE_AFTER:
            return

        print("\n[Context management: Summarizing older messages...]")

        # Calculate the number of older messages
        cut = len(self.messages) - MAX_RECENT

        # Ensure recent history begins with a user message
        while (
            cut < len(self.messages)
            and self.messages[cut]["role"] != "user"
        ):
            cut += 1

        old_messages = self.messages[:cut]
        recent_messages = self.messages[cut:]

        # Convert older messages into a transcript
        transcript = "\n".join(
            f"{m['role'].upper()}: {m['content']}"
            for m in old_messages
        )

        # Prepare summarization request
        summary_request = [
            {
                "role": "system",
                "content": SUMMARY_PROMPT
            },
            {
                "role": "user",
                "content": (
                    f"Existing summary:\n{self.summary or 'None'}\n\n"
                    f"New conversation messages:\n{transcript}"
                )
            }
        ]

        try:

            # Generate updated summary
            updated_summary = self._call_llm(
                summary_request,
                max_tokens=500
            )

            self.summary = updated_summary

            print("[Older messages summarized successfully.]")

        except ChatError as e:

            # Truncation fallback
            print("[Summary unavailable:", e, "]")
            print("[Using truncation fallback.]")

        # Keep only recent messages in active history
        self.messages = recent_messages


    # ========================================================
    # 3. BUILD COMPLETE CONVERSATION CONTEXT
    # ========================================================

    def _build_messages(self):

        # Start with system prompt
        conversation = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            }
        ]

        # Include summary of older conversation
        if self.summary:

            conversation.append({
                "role": "system",
                "content": (
                    "Summary of the earlier conversation:\n"
                    + self.summary
                    + "\nUse this information to maintain continuity."
                )
            })

        # Add recent conversation history
        conversation.extend(self.messages)

        return conversation


    # ========================================================
    # 4. SEND USER MESSAGE
    # ========================================================

    def send(self, text):

        text = (text or "").strip()

        # Validate user input
        if not text:
            raise ChatError("Please enter a message.")

        if len(text) > MAX_USER_CHARS:
            raise ChatError(
                f"Message exceeds {MAX_USER_CHARS} characters."
            )

        # Manage context before sending new message
        self._compact()

        # Add user message
        self.messages.append({
            "role": "user",
            "content": text
        })

        try:

            # Build conversation context
            conversation = self._build_messages()

            # Get AI response
            reply = self._call_llm(conversation)

            # Store assistant response
            self.messages.append({
                "role": "assistant",
                "content": reply
            })

            return reply

        except ChatError:

            # Remove failed user message
            self.messages.pop()

            raise


    # ========================================================
    # 5. RESET CONVERSATION
    # ========================================================

    def reset(self):

        self.messages = []
        self.summary = ""

        print("Conversation reset successfully!")


    # ========================================================
    # 6. DISPLAY MEMORY STATUS
    # ========================================================

    def memory_status(self):

        print("\n========== MEMORY STATUS ==========")

        print("Conversation ID:", self.conversation_id)

        print("Recent messages:", len(self.messages))

        print("Summary available:", bool(self.summary))

        if self.summary:

            print("\nRunning Summary:")
            print(self.summary)

        print("===================================")


print("Chatbot class created successfully!")

Chatbot class created successfully!


In [ ]:
# Create chatbot instance
bot = Chatbot(conversation_id="byte_001")

print("\n" + "=" * 55)
print("BYTE - YOUR AI PROGRAMMING MENTOR")
print("=" * 55)

print("""
Hello! I'm Byte.

I'm here to help you learn programming.

COMMANDS:
  reset  - Start a new conversation
  status - View conversation memory
  quit   - Exit chatbot
""")

print("=" * 55)


# Interactive conversation loop
while True:

    try:

        user_input = input("\nYou: ").strip()

        # Exit chatbot
        if user_input.lower() in ["quit", "exit"]:

            print("\nByte: Happy coding! Goodbye!")
            break


        # Reset conversation
        elif user_input.lower() == "reset":

            bot.reset()
            continue


        # View memory status
        elif user_input.lower() == "status":

            bot.memory_status()
            continue


        # Normal conversation
        reply = bot.send(user_input)

        print("\nByte:", reply)

        print(
            f"\n[Recent messages: {len(bot.messages)} | "
            f"Summary: {'Available' if bot.summary else 'Not created yet'}]"
        )


    except ChatError as e:

        print("\nERROR:", e)


    except KeyboardInterrupt:

        print("\nChat interrupted.")
        break

Conversation 'byte_001' initialized!

BYTE - YOUR AI PROGRAMMING MENTOR

Hello! I'm Byte.

I'm here to help you learn programming.

COMMANDS:
  reset  - Start a new conversation
  status - View conversation memory
  quit   - Exit chatbot


You: Explain Python variables

Byte: ## What is a Python variable?

A **variable** is just a *name* that refers to a value stored in your computer’s memory.  
Think of it like a labeled jar:

| Label | Contents |
|-------|----------|
| `age` | 27 |
| `name` | `"Ada"` |

In Python you create a variable by giving it a name and assigning a value to it with the `=` sign.

```python
age = 27          # age now refers to the number 27
name = "Ada"      # name now refers to the string "Ada"
```

You don’t have to declare the type (int, string, etc.) first—Python figures it out for you.

---

## How does assignment work?

```
x = 5          # x points to the number 5
x = "hello"    # now x points to the string "hello"
```

The variable name (`x`) is just a r